In [25]:
import pyro
import torch
from pyro.optim import SGD, Adam
import pyro.distributions as dist
from torch.distributions import constraints
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta
%matplotlib inline

import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'

## Introduction

In this notebook we the simple generative model from Slide 18, which you also experimented with in the notebook *student_BBVI.ipynb*:
 * https://www.moodle.aau.dk/mod/resource/view.php?id=1049031

In the previous notebook we derived the required gradients manually. Here we instead rely on differentiation functionality in Pyro, which, in turn is based n PyTorch.

## The model in plate notation

<img src="mean_model.png" width="600">

## The model defined in Pyro

Here we define the probabilistic model. Notice the close resemblance with the plate specification above.

In [26]:
def mean_model(data):

    # Define the random variable mu having a noral distribution as prior
    mu = pyro.sample("mu", dist.Normal(0.0,1000.0))

    # and now the plate holding the observations. The number of observations are determined by the data set 
    # supplied to the function. 
    with pyro.plate("x_plate"):
        pyro.sample(f"X", dist.Normal(mu, 1), obs=data)

## The variational distribution

In Pyro the variational distribution is defined as a so-called guide. In this example our variational distribution is a beta distribution with parameters q_alpha and q_beta:

$$
q(\mu)= \mathit{Normal}(\mu | q_{mu}, 1)
$$

In [27]:
def mean_guide(data):

    # We initialize the variational parameter to 0.0. 
    q_mu = pyro.param("q_mu", torch.tensor(0.0))

    # The name of the random variable of the variational distribution must match the name of the corresponding
    # variable in the model exactly.
    pyro.sample("mu", dist.Normal(q_mu, 1.0))

## Learning

Here we encapsulate the learning steps, relying on standard stochastic gradient descent

In [28]:
def learn(data):

    pyro.clear_param_store()

    elbo = pyro.infer.Trace_ELBO()
    svi = pyro.infer.SVI(model=mean_model,
                         guide=mean_guide,
                         optim=SGD({'lr':0.0001}),
                         loss=elbo)

    num_steps = 1000
    for step in range(num_steps):
        loss = svi.step(data)

        if step % 50 == 0:
            print(f"Loss for iteration {step}: {loss}")

In [29]:
data = torch.tensor(np.random.normal(loc=10.0, scale=1.0, size=100),dtype=torch.float)
learn(data)

Loss for iteration 0: 3627.130165576935
Loss for iteration 50: 2660.4609377384186
Loss for iteration 100: 454.4094753265381
Loss for iteration 150: 321.2006150484085
Loss for iteration 200: 194.9489208459854
Loss for iteration 250: 212.08104610443115
Loss for iteration 300: 165.13857543468475
Loss for iteration 350: 179.41643422842026
Loss for iteration 400: 207.98328518867493
Loss for iteration 450: 161.9731655716896
Loss for iteration 500: 169.14396023750305
Loss for iteration 550: 184.9688868522644
Loss for iteration 600: 176.88858795166016
Loss for iteration 650: 337.57967233657837
Loss for iteration 700: 173.96967232227325
Loss for iteration 750: 164.11788320541382
Loss for iteration 800: 162.03699678182602
Loss for iteration 850: 351.7995891571045
Loss for iteration 900: 212.48678815364838
Loss for iteration 950: 176.24404788017273


Get the learned variational parameter

## The learned parameter

In [30]:
qmu = pyro.param("q_mu").item()

In [31]:
print(f"Mean of vaiational distribution: {qmu}")

Mean of vaiational distribution: 10.079154968261719


## Exercise
* Adapt the code above to accomodate a slight more rich variational distribution, where we also have a variational parameter for the standard deviation:
$$
q(\mu)= \mathit{Normal}(\mu | q_{mu}, q_{std})
$$
* Experiment with different data sets and parameter values. Try visualizing the variational posterior distribution.

In [63]:
def mean_model(data):

    # Define the random variable mu having a noral distribution as prior
    mu = pyro.sample("mu", dist.Normal(0.0,1000.0))

    # and now the plate holding the observations. The number of observations are determined by the data set 
    # supplied to the function. 
    with pyro.plate("x_plate"):
        pyro.sample(f"X", dist.Normal(mu, 1), obs=data)

def mean_guide(data):

    # We initialize the variational parameter to 0.0. 
    q_mu = pyro.param("q_mu", torch.tensor(0.0))
    q_std = pyro.param("q_std", torch.tensor(1.0), constraint=constraints.positive)

    # The name of the random variable of the variational distribution must match the name of the corresponding
    # variable in the model exactly.
    pyro.sample("mu", dist.Normal(q_mu, q_std))


In [ ]:
qmu = pyro.param("q_mu").item()
qstd = pyro.param("q_std").item()

print(f"Mean of vaiational distribution: {qmu}")
print(f"Std of vaiational distribution: {qstd}")


Mean of vaiational distribution: 9.970426559448242
Std of vaiational distribution: 0.10008915513753891


In [69]:
def mean_model(data):

    # Define the random variable mu having a noral distribution as prior
    mu = pyro.sample("mu", dist.Normal(0.0,1000.0))
    precision = pyro.sample("prec", dist.Gamma(1,1))
    # and now the plate holding the observations. The number of observations are determined by the data set 
    # supplied to the function. 
    with pyro.plate("x_plate"):
        pyro.sample(f"X", dist.Normal(mu, 1/precision), obs=data)

def mean_guide(data):

    # We initialize the variational parameter to 0.0. 
    q_mu = pyro.param("q_mu", torch.tensor(0.0))
    q_std = pyro.param("q_std", torch.tensor(1.0), constraint=constraints.positive)

    q_a = pyro.param("q_a", torch.tensor(1.0), constraint=constraints.positive)
    q_b = pyro.param("q_b", torch.tensor(1.0), constraint=constraints.positive)

    # The name of the random variable of the variational distribution must match the name of the corresponding
    # variable in the model exactly.
    pyro.sample("mu", dist.Normal(q_mu, q_std))
    pyro.sample("prec", dist.Gamma(q_a, q_b))


In [70]:
data = torch.tensor(np.random.normal(loc=10.0, scale=1.0, size=1000),dtype=torch.float)
learn(data)

Loss for iteration 0: 7741.444880485535
Loss for iteration 50: 3773.1459491327405
Loss for iteration 100: 3737.1584141924977
Loss for iteration 150: 3630.139711126685
Loss for iteration 200: 3346.714115843177
Loss for iteration 250: 3283.8720031678677
Loss for iteration 300: 2978.6744671911
Loss for iteration 350: 2989.309271529317
Loss for iteration 400: 3165.557914197445
Loss for iteration 450: 2273.763226211071
Loss for iteration 500: 1574.3514951467514
Loss for iteration 550: 1474.5690242052078
Loss for iteration 600: 1539.8156960010529
Loss for iteration 650: 1447.470568716526
Loss for iteration 700: 1444.9169096946716
Loss for iteration 750: 1451.8415347337723
Loss for iteration 800: 1473.4445749521255
Loss for iteration 850: 1501.3510130643845
Loss for iteration 900: 1444.5800805091858
Loss for iteration 950: 1456.3529551625252


In [72]:
qmu = pyro.param("q_mu").item()
qstd = pyro.param("q_std").item()

print(f"Mean of vaiational distribution: {qmu}")
print(f"Std of vaiational distribution: {qstd}")

qa = pyro.param("q_a").item()
qb = pyro.param("q_a").item()

print(f"Mean of vaiational distribution: {qa}")
print(f"Std of vaiational distribution: {qb}")


Mean of vaiational distribution: 9.970426559448242
Std of vaiational distribution: 0.10008915513753891
Mean of vaiational distribution: 85.23138427734375
Std of vaiational distribution: 85.23138427734375
